# Advanced Pandas — Refresher + Interview Exercises

This notebook is a self-contained learning resource:

1. **Section 0** — quick refresher on pandas basics
2. **Section 1** — advanced pandas techniques (strings, regex, reshaping, grouped aggregation)
3. **Section 1b** — datetime deep-dive, focused on trading/data-engineering use cases
4. **Section 2** — worked exercise: VWAP from intraday trades (mirrors a real interview question)
5. **Section 3** — worked exercise: cleaning dirty school data + subject counts per state (mirrors a HackerRank question)
6. **Section 4** — cheat sheet

Run cells top to bottom. Sample data lives in `data/trades.csv` and `data/schools.csv`.

Each section ends with a **Practice** block: a prompt, an expected-output hint, a blank
cell for you to attempt it, and a collapsed solution cell below (click to expand). Practice
exercises use unseen data (`data/trades_practice.csv`, `data/schools_practice.csv`) or a
twist on the taught task, so you can't just copy-paste the example code.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 15)
pd.set_option("display.width", 120)


## Section 0: Basics refresher

Series, DataFrame, indexing, filtering, groupby, merge — the things you'll use every day.

In [2]:
# Series and DataFrame construction
s = pd.Series([1, 2, 3, 4], name="values")
df = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "category": ["A", "B", "A", "C", "B"],
    "amount": [10.5, 20.0, 15.25, 5.0, 30.0],
})
df.dtypes


id            int64
category        str
amount      float64
dtype: object

In [3]:
# Indexing: label-based .loc vs positional .iloc
print(df.loc[df["category"] == "A", ["id", "amount"]])
print(df.iloc[0:2])


   id  amount
0   1   10.50
2   3   15.25
   id category  amount
0   1        A    10.5
1   2        B    20.0


In [4]:
# Boolean filtering (can combine multiple conditions with & / |)
df[(df["category"] == "A") | (df["amount"] > 20.0)]


,id,category,amount
0,1,A,10.50
2,3,A,15.25
4,5,B,30.00


In [5]:
# groupby + aggregation
df.groupby("category")["amount"].agg(["sum", "mean", "count"])

df.groupby("category").agg({"amount": "mean", "id": (lambda x: (x**2).sum())})

df.groupby("category")['amount'].agg(mean="mean", square_sum=(lambda x: (x**2).sum()))


,mean,square_sum
category,,
A,12.875,342.8125
B,25.000,1300.0000
C,5.000,25.0000


In [6]:
# merge / join
categories = pd.DataFrame({"category": ["A", "B", "C"], "label": ["Alpha", "Beta", "Gamma"]})
merged = df.merge(categories, on="category", how="left")
merged.sort_values("amount", ascending=False)


,id,category,amount,label
4,5,B,30.00,Beta
1,2,B,20.00,Beta
2,3,A,15.25,Alpha
0,1,A,10.50,Alpha
3,4,C,5.00,Gamma


### Practice: Section 0

Using `practice_df` and `managers` below, test what you just learned:

1. Filter to rows where `region == "West"` **or** `units_sold > 50`.
2. Group by `region` and compute `median` and `nunique` (of `product`) — not `sum`/`mean` this time.
3. Merge `practice_df` with `managers` to attach a `manager_name` per row.

Try it in the empty cell below before expanding the solution.


In [7]:
# Starter data
practice_df = pd.DataFrame({
    "product": ["Widget", "Gadget", "Widget", "Gizmo", "Gadget"],
    "region": ["West", "East", "West", "South", "West"],
    "units_sold": [40, 60, 55, 20, 70],
})
managers = pd.DataFrame({"region": ["West", "East", "South"], "manager_name": ["Amy", "Ben", "Cara"]})
practice_df


,product,region,units_sold
0,Widget,West,40
1,Gadget,East,60
2,Widget,West,55
3,Gizmo,South,20
4,Gadget,West,70


**Expected output hint:** the grouped result should have one row per unique `region`
in your data, with a `median` column and an `nunique` column (integer count of distinct
products). The merged DataFrame should have a new `manager_name` column with no NaNs if every
region in your data exists in the lookup table.

In [8]:
# Your attempt here



In [9]:
# Solution - try it yourself first
filtered = practice_df[(practice_df["region"] == "West") | (practice_df["units_sold"] > 50)]
print(filtered)

grouped = practice_df.groupby("region")["units_sold"].agg(median="median")
grouped["distinct_products"] = practice_df.groupby("region")["product"].nunique()
print(grouped)

practice_df.merge(managers, on="region", how="left")


  product region  units_sold
0  Widget   West          40
1  Gadget   East          60
2  Widget   West          55
4  Gadget   West          70
        median  distinct_products
region                           
East      60.0                  1
South     20.0                  1
West      55.0                  2


,product,region,units_sold,manager_name
0,Widget,West,40,Amy
1,Gadget,East,60,Ben
2,Widget,West,55,Amy
3,Gizmo,South,20,Cara
4,Gadget,West,70,Amy


## Section 1: Advanced pandas techniques

String vectorization, regex cleaning, reshaping, and custom aggregation — the tools that come up
constantly once you move past toy DataFrames.

In [10]:
# Vectorized string ops via .str (avoid .apply(lambda x: ...) when a .str method exists)
names = pd.Series([" John Smith ", "jane DOE", "  bob JONES"])
names.str.strip().str.title()


0    John Smith
1      Jane Doe
2     Bob Jones
dtype: str

In [11]:
# Regex cleaning with .str.replace
messy = pd.Series(["N@Y", "C-A", "T X", "FL"])
messy.str.replace(r"[^A-Za-z0-9]", "", regex=True)


0    NY
1    CA
2    TX
3    FL
dtype: str

In [12]:
# explode: turn a space-separated string column into one row per item
sample = pd.DataFrame({"school": ["S1", "S2"], "subjects": ["Math Science", "Art"]})
sample["subjects"] = sample["subjects"].str.split()
sample.explode("subjects")


,school,subjects
0,S1,Math
0,S1,Science
1,S2,Art


In [13]:
# Custom aggregation: agg with multiple named functions, apply, and transform
grouped = df.groupby("category")["amount"]
print(grouped.agg(total="sum", spread=lambda x: x.max() - x.min()))

# transform keeps the original shape - useful for e.g. "amount as % of category total"
df["pct_of_category"] = df["amount"] / grouped.transform("sum")
df


          total  spread
category               
A         25.75    4.75
B         50.00   10.00
C          5.00    0.00


,id,category,amount,pct_of_category
0,1,A,10.50,0.407767
1,2,B,20.00,0.400000
2,3,A,15.25,0.592233
3,4,C,5.00,1.000000
4,5,B,30.00,0.600000


In [14]:
# pivot_table: reshape long -> wide with aggregation built in
pivot = df.pivot_table(index="category", values="amount", aggfunc=["sum", "count"])
pivot


,sum,count
,amount,amount
category,,
A,25.75,2
B,50.00,2
C,5.00,1


In [15]:
# Method chaining keeps transformations readable and avoids intermediate variables
result = (
    df
    .assign(amount_doubled=lambda d: d["amount"] * 2)
    .query("category != 'C'")
    .sort_values("amount_doubled", ascending=False)
)
result


,id,category,amount,pct_of_category,amount_doubled
4,5,B,30.00,0.600000,60.0
1,2,B,20.00,0.400000,40.0
2,3,A,15.25,0.592233,30.5
0,1,A,10.50,0.407767,21.0


### Practice: Section 1

Using `messy2` and `teams` below (and `practice_df` from the Section 0 starter cell):

1. Clean `messy2` with regex — strip non-alphanumeric characters and uppercase the result.
2. Split `teams["players"]` on spaces and `explode` it into one row per player.
3. Build a `pivot_table` on `practice_df` with `region` as columns instead of index, showing `sum` of `units_sold`.
4. Use `.transform` to add a column ranking `units_sold` within each `region` (hint:
   `groupby(...)["units_sold"].transform("rank")`).


In [16]:
# Starter data
messy2 = pd.Series(["N=Y", "C+A", "T&X", "  fl  "])
teams = pd.DataFrame({"team": ["T1", "T2"], "players": ["Alex Sam", "Jo Kim Lee"]})
teams


,team,players
0,T1,Alex Sam
1,T2,Jo Kim Lee


**Expected output hint:** the cleaned Series should be `["NY", "CA", "TX", "FL"]`.
The exploded `players` DataFrame should have 5 rows total (2 + 3 players). The rank column
should assign 1 to the smallest `units_sold` within each region.

In [17]:
# Your attempt here



In [18]:
# Solution - try it yourself first
cleaned = messy2.str.replace(r"[^A-Za-z0-9]", "", regex=True).str.strip().str.upper()
print(cleaned)

teams["players"] = teams["players"].str.split()
print(teams.explode("players"))

wide = practice_df.pivot_table(index="product", columns="region", values="units_sold", aggfunc="sum", fill_value=0)
print(wide)

practice_df["rank_in_region"] = practice_df.groupby("region")["units_sold"].transform("rank")
practice_df


0    NY
1    CA
2    TX
3    FL
dtype: str
  team players
0   T1    Alex
0   T1     Sam
1   T2      Jo
1   T2     Kim
1   T2     Lee
region   East  South  West
product                   
Gadget     60      0    70
Gizmo       0     20     0
Widget      0      0    95


,product,region,units_sold,rank_in_region
0,Widget,West,40,1.0
1,Gadget,East,60,1.0
2,Widget,West,55,2.0
3,Gizmo,South,20,1.0
4,Gadget,West,70,3.0


## Section 1b: Datetime deep-dive (trading-focused)

Trading data is fundamentally time-series data. This section covers the datetime tools that show
up constantly in trading/data-engineering work: parsing, timezones, market-hours filtering, bar
aggregation, trading-calendar awareness, rolling windows, and lag features.

In [19]:
# Parsing strings to datetime - the first step whenever timestamps arrive as text/CSV
raw = pd.Series(["2026-09-07 09:30:00", "2026-09-07 09:31:05", "not-a-timestamp"])
parsed = pd.to_datetime(raw, errors="coerce")  # unparseable values become NaT instead of raising
parsed


0   2026-09-07 09:30:00
1   2026-09-07 09:31:05
2                   NaT
dtype: datetime64[us]

In [20]:
# Epoch (unix) timestamps are common from upstream systems / Kafka messages
epoch_seconds = pd.Series([1757232600, 1757232660])
pd.to_datetime(epoch_seconds, unit="s")


0   2025-09-07 08:10:00
1   2025-09-07 08:11:00
dtype: datetime64[s]

In [21]:
# Timezones: tz-naive vs tz-aware, localizing and converting
ts = pd.to_datetime(pd.Series(["2026-09-07 09:30:00"]))  # naive - no tz info attached
ts_ny = ts.dt.tz_localize("America/New_York")            # tell pandas what tz the naive values ARE in
ts_utc = ts_ny.dt.tz_convert("UTC")                       # convert to a different tz
print(ts_ny.iloc[0], "->", ts_utc.iloc[0])


2026-09-07 09:30:00-04:00 -> 2026-09-07 13:30:00+00:00


In [22]:
# The .dt accessor exposes date/time components on a datetime Series
demo = pd.to_datetime(pd.Series(["2026-09-07 09:30:00", "2026-09-12 16:45:00"]))
pd.DataFrame({
    "date": demo.dt.date,
    "time": demo.dt.time,
    "hour": demo.dt.hour,
    "day_name": demo.dt.day_name(),
    "is_month_end": demo.dt.is_month_end,
})


,date,time,hour,day_name,is_month_end
0,2026-09-07,09:30:00,9,Monday,False
1,2026-09-12,16:45:00,16,Saturday,False


In [23]:
# Market-hours filtering: boolean mask on .dt.time, or between_time with a DatetimeIndex
mask = (demo.dt.time >= pd.Timestamp("08:00").time()) & (demo.dt.time <= pd.Timestamp("16:30").time())
print(mask)

indexed = pd.DataFrame({"price": [190.1, 191.4]}, index=demo)
indexed.between_time("08:00", "16:30")


0     True
1    False
dtype: bool


,price
2026-09-07 09:30:00,190.1


In [24]:
# resample: bucket a DatetimeIndex into fixed-width bars (e.g. OHLC candles)
rng = pd.date_range("2026-09-07 09:30:00", periods=20, freq="1min")
prices = pd.Series(190 + np.cumsum(np.random.default_rng(0).normal(0, 0.1, len(rng))), index=rng)

bars = prices.resample("5min").ohlc()
bars


,open,high,low,close
2026-09-07 09:30:00,190.012573,190.073895,189.999363,190.020328
2026-09-07 09:35:00,190.056487,190.281595,190.056487,190.084680
2026-09-07 09:40:00,190.022352,190.026485,189.647512,189.647512
2026-09-07 09:45:00,189.574285,189.633643,189.488229,189.633643


In [25]:
# groupby(pd.Grouper(...)) is equivalent to resample but works on a column instead of the index
df_time = pd.DataFrame({"ts": rng, "price": prices.values})
df_time.groupby(pd.Grouper(key="ts", freq="5min"))["price"].agg(["first", "max", "min", "last"])


,first,max,min,last
ts,,,,
2026-09-07 09:30:00,190.012573,190.073895,189.999363,190.020328
2026-09-07 09:35:00,190.056487,190.281595,190.056487,190.084680
2026-09-07 09:40:00,190.022352,190.026485,189.647512,189.647512
2026-09-07 09:45:00,189.574285,189.633643,189.488229,189.633643


In [26]:
# Business-day / trading-calendar awareness - weekends aren't trading days
bdays = pd.bdate_range("2026-09-07", "2026-09-18")
print(bdays)

# offsets let you shift by trading days instead of calendar days
next_bday = pd.Timestamp("2026-09-11") + pd.offsets.BDay(1)  # Friday -> Monday, skips the weekend
next_bday


DatetimeIndex(['2026-09-07', '2026-09-08', '2026-09-09', '2026-09-10', '2026-09-11', '2026-09-14', '2026-09-15',
               '2026-09-16', '2026-09-17', '2026-09-18'],
              dtype='datetime64[us]', freq='B')


Timestamp('2026-09-14 00:00:00')

In [27]:
# Rolling windows: moving averages over time - a precursor to VWAP-style calcs
prices_df = prices.to_frame("price")
prices_df["rolling_mean_5"] = prices_df["price"].rolling(window=5).mean()
prices_df["rolling_mean_2min"] = prices_df["price"].rolling("2min").mean()  # time-based window
prices_df.head(8)


,price,rolling_mean_5,rolling_mean_2min
2026-09-07 09:30:00,190.012573,NaN,190.012573
2026-09-07 09:31:00,189.999363,NaN,190.005968
2026-09-07 09:32:00,190.063405,NaN,190.031384
2026-09-07 09:33:00,190.073895,NaN,190.068650
2026-09-07 09:34:00,190.020328,190.033913,190.047111
2026-09-07 09:35:00,190.056487,190.042695,190.038408
2026-09-07 09:36:00,190.186887,190.080200,190.121687
2026-09-07 09:37:00,190.281595,190.123839,190.234241


In [28]:
# shift / diff for lag features - price change, time since last trade, etc.
lag_demo = prices.to_frame("price")
lag_demo["prev_price"] = lag_demo["price"].shift(1)
lag_demo["price_change"] = lag_demo["price"].diff()
lag_demo["seconds_since_prev"] = lag_demo.index.to_series().diff().dt.total_seconds()
lag_demo.head()


,price,prev_price,price_change,seconds_since_prev
2026-09-07 09:30:00,190.012573,NaN,NaN,NaN
2026-09-07 09:31:00,189.999363,190.012573,-0.013210,60.0
2026-09-07 09:32:00,190.063405,189.999363,0.064042,60.0
2026-09-07 09:33:00,190.073895,190.063405,0.010490,60.0
2026-09-07 09:34:00,190.020328,190.073895,-0.053567,60.0


### Practice: Section 1b (unseen data: `trades_practice.csv`)

Using `practice_trades` below (loaded from `data/trades_practice.csv`, a different week/
instrument set than the taught example):

1. Localize the timestamps to `"America/Los_Angeles"` (assume that's the exchange tz) and
   convert to `"Asia/Tokyo"`.
2. Filter to trades between **09:30** and **16:00** (a different window than the 08:00-16:30
   used earlier).
3. Resample to **15-minute** OHLC bars per instrument.
4. Compute a 10-period rolling mean of `price` per instrument, and a `diff()` lag feature.


In [29]:
# Starter data
practice_trades = pd.read_csv("data/trades_practice.csv")
practice_trades["timestamp"] = pd.to_datetime(practice_trades["timestamp"])
practice_trades.head()


,timestamp,instrument,price,volume
0,2026-09-14 07:00:00,NVDA,479.65,1285
1,2026-09-14 07:04:00,NVDA,479.95,643
2,2026-09-14 07:09:00,NVDA,479.14,4824
3,2026-09-14 07:10:00,NVDA,479.96,1808
4,2026-09-14 07:11:00,NVDA,479.13,3475


**Expected output hint:** after the 09:30-16:00 filter, row count should drop
noticeably from the raw file (raw file has ~2844 rows across 3 instruments/5 days). The
15-minute OHLC bars should have `open`/`high`/`low`/`close` columns. The Tokyo-converted
timestamps should be ~16-17 hours ahead of the Los Angeles ones (varies with DST).

In [30]:
# Your attempt here



In [31]:
# Solution - try it yourself first
la_ts = practice_trades["timestamp"].dt.tz_localize("America/Los_Angeles")
tokyo_ts = la_ts.dt.tz_convert("Asia/Tokyo")
print(la_ts.iloc[0], "->", tokyo_ts.iloc[0])

window_mask = (practice_trades["timestamp"].dt.time >= pd.Timestamp("09:30").time()) & (
    practice_trades["timestamp"].dt.time <= pd.Timestamp("16:00").time()
)
practice_session = practice_trades[window_mask].copy()
print(f"{len(practice_trades)} rows -> {len(practice_session)} rows after 09:30-16:00 filter")

ohlc_bars = (
    practice_session.set_index("timestamp")
    .groupby("instrument")["price"]
    .resample("15min")
    .ohlc()
)
print(ohlc_bars.head())

practice_session = practice_session.sort_values(["instrument", "timestamp"])
practice_session["rolling_mean_10"] = practice_session.groupby("instrument")["price"].transform(
    lambda s: s.rolling(10).mean()
)
practice_session["price_diff"] = practice_session.groupby("instrument")["price"].diff()
practice_session.head()


2026-09-14 07:00:00-07:00 -> 2026-09-14 23:00:00+09:00
2843 rows -> 1666 rows after 09:30-16:00 filter
                                  open    high     low   close
instrument timestamp                                          
AMD        2026-09-14 09:30:00  169.53  172.18  169.53  172.18
           2026-09-14 09:45:00  171.53  171.88  171.09  171.88
           2026-09-14 10:00:00  171.99  171.99  170.04  170.04
           2026-09-14 10:15:00  169.53  170.15  169.38  169.73
           2026-09-14 10:30:00  169.73  169.73  168.58  168.58


,timestamp,instrument,price,volume,rolling_mean_10,price_diff
235,2026-09-14 09:32:00,AMD,169.53,710,NaN,NaN
236,2026-09-14 09:33:00,AMD,170.39,3415,NaN,0.86
237,2026-09-14 09:37:00,AMD,170.62,1185,NaN,0.23
238,2026-09-14 09:43:00,AMD,171.36,4027,NaN,0.74
239,2026-09-14 09:44:00,AMD,172.18,4556,NaN,0.82


## Section 2: Exercise 1 — VWAP from intraday trades

**Interview scenario:** load trade data from a CSV, filter the timestamp column (initially a
string) for the regular trading session 08:00–16:30, then compute VWAP per instrument for each
day and for the whole week.

$$VWAP = \frac{\sum(price \times volume)}{\sum(volume)}$$

In [32]:
trades = pd.read_csv("data/trades.csv")
trades.head()


,timestamp,instrument,price,volume
0,2026-09-07 07:00:00,AAPL,190.14,254
1,2026-09-07 07:06:00,AAPL,189.91,1878
2,2026-09-07 07:08:00,AAPL,190.15,4517
3,2026-09-07 07:09:00,AAPL,190.24,310
4,2026-09-07 07:10:00,AAPL,189.84,1955


In [33]:
# timestamp starts out as a plain string - this is the first thing to fix
print(trades.dtypes)
trades["timestamp"] = pd.to_datetime(trades["timestamp"])
trades.dtypes


timestamp         str
instrument        str
price         float64
volume          int64
dtype: object


timestamp     datetime64[us]
instrument               str
price                float64
volume                 int64
dtype: object

In [34]:
# Filter to the regular trading session (08:00-16:30) BEFORE computing VWAP -
# pre-market/after-hours prints would otherwise skew the volume-weighted average
session_mask = (trades["timestamp"].dt.time >= pd.Timestamp("08:00").time()) & (
    trades["timestamp"].dt.time <= pd.Timestamp("16:30").time()
)
session_trades = trades[session_mask].copy()
session_trades["date"] = session_trades["timestamp"].dt.date
print(f"{len(trades)} rows -> {len(session_trades)} rows after session filter")
session_trades.head()


2846 rows -> 2171 rows after session filter


,timestamp,instrument,price,volume,date
21,2026-09-07 08:01:00,AAPL,189.58,634,2026-09-07
22,2026-09-07 08:06:00,AAPL,189.72,4425,2026-09-07
23,2026-09-07 08:12:00,AAPL,189.46,3836,2026-09-07
24,2026-09-07 08:16:00,AAPL,189.23,4612,2026-09-07
25,2026-09-07 08:18:00,AAPL,189.42,508,2026-09-07


In [35]:
# VWAP per instrument, per day
session_trades["pv"] = session_trades["price"] * session_trades["volume"]

daily_vwap = (
    session_trades.groupby(["date", "instrument"])
    .apply(lambda g: g["pv"].sum() / g["volume"].sum(), include_groups=False)
    .rename("vwap")
    .reset_index()
)
daily_vwap


,date,instrument,vwap
0,2026-09-07,AAPL,188.285059
1,2026-09-07,GOOG,163.856568
2,2026-09-07,MSFT,417.019312
3,2026-09-08,AAPL,188.137332
4,2026-09-08,GOOG,165.250122
5,2026-09-08,MSFT,418.003154
6,2026-09-09,AAPL,191.667019
7,2026-09-09,GOOG,165.319849
8,2026-09-09,MSFT,417.103467
9,2026-09-10,AAPL,189.449869


In [36]:
# VWAP per instrument for the whole week (same formula, no date grouping)
weekly_vwap = (
    session_trades.groupby("instrument")
    .apply(lambda g: g["pv"].sum() / g["volume"].sum(), include_groups=False)
    .rename("weekly_vwap")
    .reset_index()
)
weekly_vwap


,instrument,weekly_vwap
0,AAPL,190.196783
1,GOOG,164.458865
2,MSFT,416.893952


### Practice: Section 2 (unseen data + a twist)

Using `practice_trades2` below (loaded + timestamp-parsed from `data/trades_practice.csv`):

1. Filter to the regular session (08:00-16:30), as before.
2. **Twist:** additionally exclude the first and last 15 minutes of the session (08:00-08:15
   and 16:15-16:30), since VWAP is sometimes computed on the "stable" part of the day only.
3. Compute daily VWAP per instrument, and weekly VWAP per instrument, on this narrower window.


In [37]:
# Starter data
practice_trades2 = pd.read_csv("data/trades_practice.csv")
practice_trades2["timestamp"] = pd.to_datetime(practice_trades2["timestamp"])
practice_trades2.head()


,timestamp,instrument,price,volume
0,2026-09-14 07:00:00,NVDA,479.65,1285
1,2026-09-14 07:04:00,NVDA,479.95,643
2,2026-09-14 07:09:00,NVDA,479.14,4824
3,2026-09-14 07:10:00,NVDA,479.96,1808
4,2026-09-14 07:11:00,NVDA,479.13,3475


**Expected output hint:** the narrowed session should have fewer rows than the plain
08:00-16:30 filter. You should still get one weekly VWAP value per instrument (3 rows: NVDA,
AMD, NFLX), each a plausible price near that instrument's base level.

In [38]:
# Your attempt here



In [39]:
# Solution - try it yourself first
t = practice_trades2["timestamp"].dt.time
session_mask = (t >= pd.Timestamp("08:00").time()) & (t <= pd.Timestamp("16:30").time())
narrow_mask = (t >= pd.Timestamp("08:15").time()) & (t <= pd.Timestamp("16:15").time())
narrowed = practice_trades2[session_mask & narrow_mask].copy()
narrowed["date"] = narrowed["timestamp"].dt.date
narrowed["pv"] = narrowed["price"] * narrowed["volume"]

daily_vwap_practice = (
    narrowed.groupby(["date", "instrument"])
    .apply(lambda g: g["pv"].sum() / g["volume"].sum(), include_groups=False)
    .rename("vwap")
    .reset_index()
)
print(daily_vwap_practice)

weekly_vwap_practice = (
    narrowed.groupby("instrument")
    .apply(lambda g: g["pv"].sum() / g["volume"].sum(), include_groups=False)
    .rename("weekly_vwap")
    .reset_index()
)
weekly_vwap_practice


          date instrument        vwap
0   2026-09-14        AMD  168.127341
1   2026-09-14       NFLX  618.670814
2   2026-09-14       NVDA  481.724363
3   2026-09-15        AMD  170.750543
4   2026-09-15       NFLX  613.768317
5   2026-09-15       NVDA  483.485229
6   2026-09-16        AMD  168.383028
7   2026-09-16       NFLX  609.249201
8   2026-09-16       NVDA  482.562589
9   2026-09-17        AMD  162.853869
10  2026-09-17       NFLX  620.612795
11  2026-09-17       NVDA  479.020017
12  2026-09-18        AMD  165.074029
13  2026-09-18       NFLX  626.413239
14  2026-09-18       NVDA  475.296814


,instrument,weekly_vwap
0,AMD,167.270862
1,NFLX,617.624811
2,NVDA,480.348992


## Section 3: Exercise 2 — dirty school/state/subjects data

**Interview scenario:** a DataFrame has `school_code`, `state_code` (dirty, random special chars),
and `subjects` (a space-separated string list). Remove schools with fewer than 3 subjects, clean
`state_code` to alphanumeric-only, then produce a count of a given set of target subjects per
`state_code`.

In [40]:
schools = pd.read_csv("data/schools.csv")
schools


,school_code,state_code,subjects
0,SCH001,N_Y,History
1,SCH002,FL,Math
2,SCH003,T X,Science Music Art PE English CS
3,SCH004,W@A,Music Science History Math English
4,SCH005,F#L,Art Science Music English
...,...,...,...
35,SCH036,N-Y,Science Art History CS Math Music
36,SCH037,FL,PE CS Music
37,SCH038,N$Y,Science Math Art
38,SCH039,NY,Art


In [41]:
# Split the space-separated subjects string into a list, and count subjects per school
schools["subjects_list"] = schools["subjects"].str.split()
schools["subject_count"] = schools["subjects_list"].str.len()
schools.head()


,school_code,state_code,subjects,subjects_list,subject_count
0,SCH001,N_Y,History,[History],1
1,SCH002,FL,Math,[Math],1
2,SCH003,T X,Science Music Art PE English CS,"[Science, Music, Art, PE, English, CS]",6
3,SCH004,W@A,Music Science History Math English,"[Music, Science, History, Math, English]",5
4,SCH005,F#L,Art Science Music English,"[Art, Science, Music, English]",4


In [42]:
# Drop schools with fewer than 3 subjects
filtered = schools[schools["subject_count"] >= 3].copy()
print(f"{len(schools)} schools -> {len(filtered)} schools after the >=3 subjects filter")


40 schools -> 34 schools after the >=3 subjects filter


In [43]:
# Clean state_code: strip everything that isn't alphanumeric
filtered["state_code_clean"] = filtered["state_code"].str.replace(r"[^A-Za-z0-9]", "", regex=True)
filtered[["school_code", "state_code", "state_code_clean"]].head()


,school_code,state_code,state_code_clean
2,SCH003,T X,TX
3,SCH004,W@A,WA
4,SCH005,F#L,FL
5,SCH006,N!Y,NY
6,SCH007,N_Y,NY


In [44]:
# Explode subjects so each (school, subject) pair is its own row
exploded = filtered.explode("subjects_list").rename(columns={"subjects_list": "subject"})
exploded.head()


,school_code,state_code,subjects,subject,subject_count,state_code_clean
2,SCH003,T X,Science Music Art PE English CS,Science,6,TX
2,SCH003,T X,Science Music Art PE English CS,Music,6,TX
2,SCH003,T X,Science Music Art PE English CS,Art,6,TX
2,SCH003,T X,Science Music Art PE English CS,PE,6,TX
2,SCH003,T X,Science Music Art PE English CS,English,6,TX


In [45]:
# Final output: count of each target subject per cleaned state_code
target_subjects = ["Math", "Science", "English"]

subject_counts = (
    exploded[exploded["subject"].isin(target_subjects)]
    .pivot_table(index="state_code_clean", columns="subject", aggfunc="size", fill_value=0)
    .reindex(columns=target_subjects, fill_value=0)
)
subject_counts


subject,Math,Science,English
state_code_clean,,,
CA,1,0,0
FL,2,3,2
NY,9,7,6
TX,6,6,6
WA,2,1,2


### Practice: Section 3 (unseen data + a twist)

Using `practice_schools` below (loaded from `data/schools_practice.csv`, different dirty
characters and subjects/states than the taught example):

1. Split `subjects`, count them, and drop schools with **fewer than 4** subjects (a different
   threshold than the `< 3` used earlier).
2. Clean `state_code` to alphanumeric only.
3. Explode subjects, then count a **different** target subject list: `["Math", "CS", "Art"]`.
4. **Twist:** add a `Total` column to the output that sums the target-subject counts per state.


In [46]:
# Starter data
practice_schools = pd.read_csv("data/schools_practice.csv")
practice_schools


,school_code,state_code,subjects
0,PSCH001,C*O,History Math English Science Music Economics Art
1,PSCH002,N^V,History Math
2,PSCH003,NV,Math Geography Economics CS
3,PSCH004,OR,Music Art Math CS PE Geography History
4,PSCH005,.NV.,Art Geography Music Math English Economics CS
...,...,...,...
30,PSCH031,C%O,Music Economics Science Math
31,PSCH032,N~V,CS Art PE Geography Science History
32,PSCH033,O/R,English Art CS Economics Music History
33,PSCH034,CO,History


**Expected output hint:** the cleaned `state_code` values should all be 2-letter
codes from `{OR, NV, AZ, CO, UT}`. Your final table should have one row per state, three
subject columns (Math, CS, Art), and a `Total` column equal to the row sum.

In [47]:
# Your attempt here



In [48]:
# Solution - try it yourself first
practice_schools["subjects_list"] = practice_schools["subjects"].str.split()
practice_schools["subject_count"] = practice_schools["subjects_list"].str.len()

filtered_practice = practice_schools[practice_schools["subject_count"] >= 4].copy()
filtered_practice["state_code_clean"] = filtered_practice["state_code"].str.replace(
    r"[^A-Za-z0-9]", "", regex=True
)

exploded_practice = filtered_practice.explode("subjects_list").rename(columns={"subjects_list": "subject"})

target_subjects_2 = ["Math", "CS", "Art"]
counts_practice = (
    exploded_practice[exploded_practice["subject"].isin(target_subjects_2)]
    .pivot_table(index="state_code_clean", columns="subject", aggfunc="size", fill_value=0)
    .reindex(columns=target_subjects_2, fill_value=0)
)
counts_practice["Total"] = counts_practice.sum(axis=1)
counts_practice


subject,Math,CS,Art,Total
state_code_clean,,,,
AZ,4,4,3,11
CO,6,2,4,12
NV,6,7,4,17
OR,2,2,4,8
UT,1,2,0,3


## Section 4: Cheat sheet

| Task | Method |
|---|---|
| Parse string to datetime | `pd.to_datetime(series, errors="coerce")` |
| Localize / convert timezone | `.dt.tz_localize(tz)` / `.dt.tz_convert(tz)` |
| Filter by time of day | `.dt.time` mask or `df.between_time(start, end)` (needs `DatetimeIndex`) |
| Bar aggregation (OHLC) | `series.resample("5min").ohlc()` or `groupby(pd.Grouper(freq=...))` |
| Business days only | `pd.bdate_range(...)`, `pd.offsets.BDay(n)` |
| Moving average | `.rolling(window=N)` or `.rolling("1D")` for time-based windows |
| Lag / diff features | `.shift(n)`, `.diff()` |
| Clean text column | `.str.replace(pattern, "", regex=True)` |
| String list -> rows | `.str.split()` then `.explode(col)` |
| Reshape long -> wide | `.pivot_table(index=..., columns=..., aggfunc=...)` |
| Weighted average pattern (VWAP) | `groupby(...).apply(lambda g: (g.a * g.w).sum() / g.w.sum())` |
